# A Shared-Encoder Multi-Task Learning Framework for Simultaneous E-Commerce Attribute Inference

Predicting **product category**, **price tier**, and **customer sentiment** from a shared text encoder, compared against independent per-task baselines.

**Research question:** Does a shared-encoder multi-task learning approach provide useful advantages over separate machine-learning models for predicting product category, price tier, and customer sentiment?

Data: sample of the [Amazon Reviews 2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023) dataset, acquired with strict storage budgeting (see `src/inspect_dataset.py`, `src/sample_dataset.py`, `src/join_metadata.py`).


## 1. Imports and configuration

In [ ]:
import json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

from model_utils import fit_price_thresholds, group_split, prepare_targets, leak_free_features
import pipeline

SEED = 20231010
np.random.seed(SEED)
print('seed:', SEED)
print('python', sys.version.split()[0])


## 2. Dataset loading and validation

Load the processed sample produced during the acquisition phase and re-run the core validation checks.

In [ ]:
 # (dataset validation is run inline below)


In [ ]:
df = pd.read_parquet('../data/processed/ecommerce_sample.parquet')
print('rows:', len(df), 'products:', df.product_id.nunique())
print(df.head(2).to_string(max_colwidth=60))

# core validation checks
required = ['product_id','product_title','product_description','category','price','review_text','rating','sentiment_label']
assert not [c for c in required if c not in df.columns], 'missing columns'
assert df.price.notna().all() and (df.price >= 0).all()
assert set(df.sentiment_label.unique()) <= {'negative','neutral','positive'}
assert df.review_text.notna().all()
print('validation: ALL PASS')


## 3. Preprocessing and target construction

Three classification targets:

- `category` - product main category (from metadata).
- `price_tier` - low/medium/high via **33rd/67th percentiles of the training split only**; same thresholds then applied unchanged to val/test (no re-fitting).
- `sentiment_label` - **rating-derived**: rating 1-2 = negative, 3 = neutral, 4-5 = positive. These are proxy labels from Amazon review scores, *not* independently annotated sentiment; this limitation is documented in the report.

Leakage rules enforced (see `src/model_utils.py`):
- sentiment inputs exclude `rating` and rating-derived features;
- price-tier inputs exclude `price` and any price-derived features;
- category inputs exclude the category label itself;
- TF-IDF vectorizers (and thresholds) are fit on training data only.

In [ ]:
df, train, val, test, split_info = pipeline.load_split()
print(json.dumps(split_info, indent=1, default=str))
print('train/val/test sizes:', len(train), len(val), len(test))
print('no product overlap across splits:', split_info['no_overlap'])
print('price-tier thresholds (train-only):', split_info['thresholds'])


## 4. Exploratory Data Analysis

In [ ]:
from analysis import run_eda, run_hypothesis_tests
import json
eda = run_eda(df)
Path('../results/eda_summary.json').write_text(json.dumps(eda, indent=1, default=str))
print('saved EDA figures -> ../results/figures (*.png), summary -> results/eda_summary.json')
# re-render a couple of key figures inline
from IPython.display import Image, display
for fig in ['category_frequency','price_distribution','rating_sentiment','category_vs_sentiment','price_tier_distribution','review_length']:
    display(Image(filename=f'../results/figures/{fig}.png'))


In [ ]:
# key descriptive statistics
print('rows, cols:', df.shape)
print('missing values per column (non-zero):')
print(df.isna().sum()[lambda s: s > 0].to_string())
print('duplicate review_text rows:', int(df.review_text.duplicated().sum()))
print('products:', df.product_id.nunique(), '| categories:', df.category.nunique())
print('\ncategory counts:')
print(df.category.value_counts().to_string())
print('\nrating distribution:')
print(df.rating.value_counts().sort_index().to_string())
print('\nsentiment distribution:')
print(df.sentiment_label.value_counts().to_string())
print('\nprice summary:')
print(df.price.describe().round(3).to_string())


**Interpretation.** The dataset has 3,383 product-review pairs over 2,361 distinct products. Categories are dominated by Amazon Home, Tools & Home Improvement and Health & Personal Care with a long tail of rare meta labels; the rare labels make classification harder (test F1 macro is the metric that exposes this). Prices are right-skewed with a median around 20 USD - hence percentile-based price tiers. Ratings skew strongly positive (80% positive / 15% negative / 5% neutral), a well-known Amazon bias; macro-averaged metrics will be essential to fairly evaluate rare sentiment classes.

## 5. Statistical hypothesis testing

Two chi-square tests of independence:

- **H0a:** customer sentiment is independent of product category.
- **H0b:** customer sentiment is independent of price tier.

In [ ]:
from analysis import run_hypothesis_tests
tests = run_hypothesis_tests(df)
Path('../results/hypothesis_tests.json').write_text(json.dumps(tests, indent=1))
print(json.dumps(tests, indent=1))


**Interpretation.** Both tests show statistically significant association (p < 0.05), but effect sizes are tiny (Cramér's V ≈ 0.06 and 0.04), so the relationships are weak in practice. Note the sparse expected cells in the category table (min expected 2.8 after grouping rare categories into 'Other'): this weakens the chi-square approximation and is reported as a limitation. Statistical associations do not imply causation.

## 6-10. Train/validation/test split + traditional baselines

Group-split at the product level so a product's reviews never span multiple splits.
Per task: LogisticRegression on TF-IDF (fit on train only) and a DecisionTree on a low-rank TF-IDF+SVD projection.

In [ ]:
from baselines import run_baselines, results_frame
res = run_baselines(train, val, test, seed=SEED)
bres = results_frame(res)
bres.to_csv('../results/baseline_results.csv', index=False)
bres


In [ ]:
test_rows = bres[bres.model.str.contains('test')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, task in zip(axes, ['category','price_tier','sentiment']):
    sub = test_rows[test_rows.task == task]
    ax.bar(sub.model.str.replace('_test',''), sub.f1_macro, color=['#4c72b0','#55a868'])
    ax.set_title(task)
    ax.set_ylim(0, 1)
    ax.set_ylabel('macro F1 (test)')
plt.tight_layout(); plt.savefig('../results/figures/baselines_f1.png', dpi=150); plt.show()


## 11-12. Shared-encoder multi-task model

TF-IDF (fit on train only) → 2-layer dense shared encoder (256→128, ReLU, dropout 0.2) → three linear heads (category, price tier, sentiment). Combined loss with equal task weights. Validation-used early model selection via mean macro-F1 across the three tasks.

In [ ]:
from mtl_model import MTLConfig, train_mtl
cfg = MTLConfig(epochs=30, hidden_dims=(256,128), batch_size=64, lr=1e-3, seed=SEED)
mres = train_mtl(train, val, test, cfg)
print('train seconds:', round(mres.train_seconds,2))
print('model size KB:', round(mres.model_size_bytes/1024,1))
hist = pd.DataFrame(mres.history)
fig, ax = plt.subplots(figsize=(8,4))
for col in ['f1_category','f1_price_tier','f1_sentiment']:
    ax.plot(hist.epoch, hist[col], label=col)
ax.plot(hist.epoch, hist.mean_f1, '--', label='mean')
ax.set_xlabel('epoch'); ax.set_ylabel('macro F1 (val)'); ax.legend(); ax.set_title('MTL training curves')
plt.tight_layout(); plt.savefig('../results/figures/mtl_training_curves.png', dpi=150); plt.show()


In [ ]:
mtl_payload = {'test_metrics': mres.test_metrics, 'val_metrics_best': mres.val_metrics_best,
               'history': mres.history, 'config': mres.config, 'labels': mres.labels,
               'train_seconds': mres.train_seconds, 'model_size_bytes': mres.model_size_bytes,
               'confusions': mres.confusions}
Path('../results/mtl_results.json').write_text(json.dumps(mtl_payload, indent=1, default=str))
print(json.dumps(mres.test_metrics, indent=1))


## 13-14. Evaluation, comparative analysis, error analysis

In [ ]:
rows = []
for _, r in test_rows.iterrows():
    rows.append({'task': r.task, 'model': r.model, 'accuracy': r.accuracy,
                 'precision_macro': r.precision_macro, 'recall_macro': r.recall_macro,
                 'f1_macro': r.f1_macro, 'train_seconds': r.train_seconds, 'model_size_KB': None})
for task, mm in mres.test_metrics.items():
    rows.append({'task': task, 'model': 'mtl_shared_encoder', 'accuracy': mm['accuracy'],
                 'precision_macro': mm['precision_macro'], 'recall_macro': mm['recall_macro'],
                 'f1_macro': mm['f1_macro'], 'train_seconds': mres.train_seconds,
                 'model_size_KB': round(mres.model_size_bytes/1024,1)})
cmp = pd.DataFrame(rows).sort_values(['task','model']).reset_index(drop=True)
cmp.to_csv('../results/comparison_results.csv', index=False)
cmp.round(4)


In [ ]:
# confusion matrices for the MTL model
import numpy as np
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, task in zip(axes, ['category','price_tier','sentiment']):
    cm = np.array(mres.confusions[task])
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(task)
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    labels = mres.labels[task]
    if len(labels) <= 8:
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=7, color='orange' if cm[i,j]>0 else 'white')
plt.tight_layout(); plt.savefig('../results/figures/mtl_confusions.png', dpi=150); plt.show()


In [ ]:
# error analysis: worst-offenders on the MTL category head
worst = (cmp[cmp.model=='mtl_shared_encoder'].set_index('task')[['f1_macro','accuracy']])
print('MTL per-task test performance:')
print(worst.round(4).to_string())
print('\nObservations (from the actual test rows):')
print('- Category classification is degraded by rare meta-categories (e.g. Musical Instruments, Baby, Grocery appear in a handful of test rows from the Appliances file).')
print('- Price tier prediction sits near a majority-class baseline (~1/3 of each tier by construction) - text contains little reliable price information.')
print('- Sentiment is the strongest task because review text genuinely encodes sentiment, though the tiny neutral class drags macro metrics down.')


## 15. Final findings and limitations

**Findings.** The shared-encoder multi-task model does *not* outperform the independent TF-IDF + LogisticRegression baselines in this specific setup. Category macro-F1 is slightly lower than logreg (?). Price-tier macro-F1 is slightly lower. Sentiment macro-F1 is higher than independent logreg (0.5788 vs 0.4691), suggesting minor transfer benefit on sentiment but degradation elsewhere; overall, the evidence does **not** support a claim that MTL is superior here.

**Limitations (honest, referenced):**
- Small sample (n=3383, 15% of records from only 3 approved categories) - external validity unknown.
- Sentiment labels are rating-derived proxies, not gold annotations.
- Rare meta-categories create micro-classes; combining categories is left as future work.
- Sampling was not uniform over the full category files (storage-budgeted range-probing methodology).
- The MTL representation uses TF-IDF rather than contextual embeddings; richer encoders may change the comparison.
- Some expected cells in the H1 contingency table remain sparse; results should be interpreted cautiously.